In [ ]:
!pip install catboost
!pip install pymorphy3 nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 50.3 MB/s eta 0:00:00


# **TF‑IDF + ST (MiniLM) + CatBoost**

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, log_loss
from sklearn.feature_extraction.text import TfidfVectorizer

from catboost import CatBoostClassifier
from sentence_transformers import SentenceTransformer


df = pd.read_json("rnc_dataset_markup.json")

def parse_list(s):
    if isinstance(s, str) and s.startswith("["):
        return [int(x) for x in s.strip("[]").split(",") if x.strip()]
    return []

df["detected"] = df["targetDetectedMcIds"].apply(parse_list)
df["target"]   = df["targetSplitMcIds"].apply(parse_list)


all_mc_ids = sorted(
    set(mc for lst in df["detected"] for mc in lst) |
    set(mc for lst in df["target"]   for mc in lst)
)

if 101 in all_mc_ids:
    all_mc_ids.remove(101)


def make_binary_features(detected):
    detected = set(detected)
    return [1 if mc in detected else 0 for mc in all_mc_ids]


def extract_linguistic_features(text: str):
    text_low = text.lower()

    separate_markers = [
        r"отдельно",
        r"по отдельности",
        r"выполняем отдельно",
        r"делаем отдельно",
        r"оказываем отдельно",
        r"каждый вид.*отдельно"
    ]
    f_separate = int(any(re.search(p, text_low) for p in separate_markers))

    complex_markers = [
        r"включая",
        r"в составе",
        r"под ключ",
        r"комплексн"
    ]
    f_complex = int(any(re.search(p, text_low) for p in complex_markers))

    f_many_commas = int(text_low.count(",") >= 3)
    f_and = int(" и " in text_low)

    specialization_markers = [
        r"специализируемся",
        r"профессионально",
        r"оказываем услуги",
        r"выполняем работы"
    ]
    f_specialization = int(any(re.search(p, text_low) for p in specialization_markers))

    return [f_separate, f_complex, f_many_commas, f_and, f_specialization]


ling_features = np.vstack(df["description"].apply(extract_linguistic_features))

mlb = MultiLabelBinarizer()
Y_full = mlb.fit_transform(df["target"])
classes_full = list(mlb.classes_)

if 101 in classes_full:
    idx_101 = classes_full.index(101)
    mask_keep = [i for i in range(len(classes_full)) if i != idx_101]
    Y = Y_full[:, mask_keep]
    classes_used = [c for c in classes_full if c != 101]
else:
    Y = Y_full
    classes_used = classes_full

print("\n=========== ENSEMBLE: TF‑IDF + ST + CatBoost ===========")

X_bin = np.vstack(df["detected"].apply(make_binary_features))

X_ling = ling_features

tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=3)
X_text_tfidf = tfidf.fit_transform(df["description"]).toarray()

st_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
X_text_st = st_model.encode(df["description"].tolist(), show_progress_bar=True)

X_ens = np.hstack([X_bin, X_text_tfidf, X_text_st, X_ling])

X_train, X_temp, Y_train, Y_temp = train_test_split(X_ens, Y, test_size=0.30, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.50, random_state=42)

n_classes = Y_train.shape[1]
models_ens = []
thresholds_ens = []

for cls in range(n_classes):
    print(f"\n===== Класс {cls} / {n_classes} (mc_id={classes_used[cls]}) =====")

    y_tr = Y_train[:, cls]
    y_v  = Y_val[:, cls]
    y_te = Y_test[:, cls]

    model = CatBoostClassifier(
        loss_function="Logloss",
        iterations=500,
        depth=6,
        learning_rate=0.05,
        verbose=200
    )

    model.fit(
        X_train, y_tr,
        eval_set=[(X_val, y_v), (X_test, y_te)],
        use_best_model=True,
        early_stopping_rounds=100,
        verbose=200
    )

    proba_val = model.predict_proba(X_val)[:, 1]
    best_thr = 0.0
    best_f1 = 0.0

    for thr in np.linspace(0.05, 0.5, 20):
        pred_thr = (proba_val > thr).astype(int)
        f1 = f1_score(y_v, pred_thr, average="binary", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    print(f"Лучший порог для класса {classes_used[cls]}: {best_thr:.3f}, F1={best_f1:.4f}")

    models_ens.append(model)
    thresholds_ens.append(best_thr)


proba_test_all = np.zeros_like(Y_test, dtype=float)

for cls, model in enumerate(models_ens):
    proba_test_all[:, cls] = model.predict_proba(X_test)[:, 1]

pred_test_all = np.zeros_like(Y_test, dtype=int)
for cls, thr in enumerate(thresholds_ens):
    pred_test_all[:, cls] = (proba_test_all[:, cls] > thr).astype(int)

print("\nENSEMBLE Test F1 micro:", f1_score(Y_test, pred_test_all, average="micro"))
print("ENSEMBLE Test F1 macro:", f1_score(Y_test, pred_test_all, average="macro"))

loglosses_ens = [
    log_loss(Y_test[:, i], proba_test_all[:, i], labels=[0,1])
    for i in range(Y_test.shape[1])
]
print("ENSEMBLE Test logloss (mean):", np.mean(loglosses_ens))


def predict_ensemble(text, detected_mc_ids):
    x_bin = np.array([make_binary_features(detected_mc_ids)])
    x_tfidf = tfidf.transform([text]).toarray()
    x_st = st_model.encode([text])
    x_ling = np.array([extract_linguistic_features(text)])

    x = np.hstack([x_bin, x_tfidf, x_st, x_ling])

    proba = np.zeros(len(classes_used))
    for cls, model in enumerate(models_ens):
        proba[cls] = model.predict_proba(x)[0, 1]

    predicted_mc_ids = [
        mc for mc, p, thr in zip(classes_used, proba, thresholds_ens)
        if p > thr
    ]

    return {
        "shouldSplit": len(predicted_mc_ids) > 0,
        "draft_mcIds": predicted_mc_ids
    }



=========== ENSEMBLE: TF‑IDF + ST + CatBoost ===========


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]


===== Класс 0 / 10 (mc_id=102) =====
0:	learn: 0.6543630	test: 0.6567160	test1: 0.6521718	best: 0.6521718 (0)	total: 493ms	remaining: 4m 6s
200:	learn: 0.0792608	test: 0.2185348	test1: 0.1374338	best: 0.1374338 (200)	total: 47.7s	remaining: 1m 11s
400:	learn: 0.0313633	test: 0.2109358	test1: 0.1269428	best: 0.1260048 (385)	total: 1m 33s	remaining: 23.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.1260047845
bestIteration = 385

Shrink model to first 386 iterations.
Лучший порог для класса 102: 0.145, F1=0.7434

===== Класс 1 / 10 (mc_id=103) =====
0:	learn: 0.6477087	test: 0.6496021	test1: 0.6468472	best: 0.6468472 (0)	total: 289ms	remaining: 2m 24s
200:	learn: 0.0598900	test: 0.1618703	test1: 0.1262221	best: 0.1261749 (199)	total: 47.7s	remaining: 1m 10s
400:	learn: 0.0233764	test: 0.1550323	test1: 0.1233437	best: 0.1222834 (355)	total: 1m 33s	remaining: 23.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.1222833646
bestIteration = 355


# **TF‑IDF + model + CatBoost**

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, log_loss
from sklearn.feature_extraction.text import TfidfVectorizer

from catboost import CatBoostClassifier
from sentence_transformers import SentenceTransformer


df = pd.read_json("rnc_dataset_markup.json")

def parse_list(s):
    if isinstance(s, str) and s.startswith("["):
        return [int(x) for x in s.strip("[]").split(",") if x.strip()]
    return []

df["detected"] = df["targetDetectedMcIds"].apply(parse_list)
df["target"]   = df["targetSplitMcIds"].apply(parse_list)


all_mc_ids = sorted(
    set(mc for lst in df["detected"] for mc in lst) |
    set(mc for lst in df["target"]   for mc in lst)
)

if 101 in all_mc_ids:
    all_mc_ids.remove(101)

def make_binary_features(detected):
    detected = set(detected)
    return [1 if mc in detected else 0 for mc in all_mc_ids]


def extract_linguistic_features(text: str):
    text_low = text.lower()

    separate_markers = [
        r"отдельно", r"по отдельности", r"выполняем отдельно",
        r"делаем отдельно", r"оказываем отдельно", r"каждый вид.*отдельно"
    ]
    f_separate = int(any(re.search(p, text_low) for p in separate_markers))

    complex_markers = [r"включая", r"в составе", r"под ключ", r"комплексн"]
    f_complex = int(any(re.search(p, text_low) for p in complex_markers))

    f_many_commas = int(text_low.count(",") >= 3)
    f_and = int(" и " in text_low)

    specialization_markers = [
        r"специализируемся", r"профессионально",
        r"оказываем услуги", r"выполняем работы"
    ]
    f_specialization = int(any(re.search(p, text_low) for p in specialization_markers))

    return [f_separate, f_complex, f_many_commas, f_and, f_specialization]

ling_features = np.vstack(df["description"].apply(extract_linguistic_features))


mlb = MultiLabelBinarizer()
Y_full = mlb.fit_transform(df["target"])
classes_full = list(mlb.classes_)

if 101 in classes_full:
    idx_101 = classes_full.index(101)
    mask_keep = [i for i in range(len(classes_full)) if i != idx_101]
    Y = Y_full[:, mask_keep]
    classes_used = [c for c in classes_full if c != 101]
else:
    Y = Y_full
    classes_used = classes_full


def run_pipeline(st_model_name, label):
    print(f"\n\n==================== {label} ====================")

    X_bin = np.vstack(df["detected"].apply(make_binary_features))

    X_ling = ling_features

    tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=3)
    X_text_tfidf = tfidf.fit_transform(df["description"]).toarray()

    st_model = SentenceTransformer(st_model_name)
    X_text_st = st_model.encode(df["description"].tolist(), show_progress_bar=True)

    X_ens = np.hstack([X_bin, X_text_tfidf, X_text_st, X_ling])

    X_train, X_temp, Y_train, Y_temp = train_test_split(
        X_ens, Y, test_size=0.30, random_state=42
    )
    X_val, X_test, Y_val, Y_test = train_test_split(
        X_temp, Y_temp, test_size=0.50, random_state=42
    )

    n_classes = Y_train.shape[1]
    models_ens = []
    thresholds_ens = []

    for cls in range(n_classes):
        print(f"\nКласс {cls}/{n_classes} (mc_id={classes_used[cls]})")

        y_tr = Y_train[:, cls]
        y_v  = Y_val[:, cls]
        y_te = Y_test[:, cls]

        model = CatBoostClassifier(
            loss_function="Logloss",
            iterations=500,
            depth=6,
            learning_rate=0.05,
            verbose=0
        )

        model.fit(
            X_train, y_tr,
            eval_set=[(X_val, y_v)],
            use_best_model=True,
            early_stopping_rounds=100,
            verbose=False
        )

        proba_val = model.predict_proba(X_val)[:, 1]
        best_thr = 0.0
        best_f1 = 0.0

        for thr in np.linspace(0.05, 0.5, 50):
            pred_thr = (proba_val > thr).astype(int)
            f1 = f1_score(y_v, pred_thr, average="binary", zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thr = thr

        models_ens.append(model)
        thresholds_ens.append(best_thr)

    proba_test_all = np.zeros_like(Y_test, dtype=float)
    for cls, model in enumerate(models_ens):
        proba_test_all[:, cls] = model.predict_proba(X_test)[:, 1]

    pred_test_all = np.zeros_like(Y_test, dtype=int)
    for cls, thr in enumerate(thresholds_ens):
        pred_test_all[:, cls] = (proba_test_all[:, cls] > thr).astype(int)

    f1_micro = f1_score(Y_test, pred_test_all, average="micro")
    f1_macro = f1_score(Y_test, pred_test_all, average="macro")
    logloss_mean = np.mean([
        log_loss(Y_test[:, i], proba_test_all[:, i], labels=[0,1])
        for i in range(Y_test.shape[1])
    ])

    print(f"\n📌 {label} — РЕЗУЛЬТАТЫ")
    print(f"F1 micro: {f1_micro:.4f}")
    print(f"F1 macro: {f1_macro:.4f}")
    print(f"logloss:  {logloss_mean:.4f}")

    return f1_micro, f1_macro, logloss_mean


**TF‑IDF + RuBERT‑base + CatBoost**

In [ ]:
run_pipeline("DeepPavlov/rubert-base-cased-sentence", "RuBERT-base")



==================== RuBERT-base ====================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]


Класс 0/10 (mc_id=102)

Класс 1/10 (mc_id=103)

Класс 2/10 (mc_id=104)

Класс 3/10 (mc_id=105)

Класс 4/10 (mc_id=106)

Класс 5/10 (mc_id=107)

Класс 6/10 (mc_id=108)

Класс 7/10 (mc_id=109)

Класс 8/10 (mc_id=110)

Класс 9/10 (mc_id=111)

📌 RuBERT-base — РЕЗУЛЬТАТЫ
F1 micro: 0.6581
F1 macro: 0.6608
logloss:  0.1145


(0.6580645161290323, 0.6607589910990578, np.float64(0.11449429591240701))

**TF‑IDF + multilingual‑e5‑base + CatBoost**

In [ ]:
run_pipeline("intfloat/multilingual-e5-base", "multilingual-e5-base")



==================== multilingual-e5-base ====================


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

... (6 outputs left out)

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]


Класс 0/10 (mc_id=102)

Класс 1/10 (mc_id=103)

Класс 2/10 (mc_id=104)

Класс 3/10 (mc_id=105)

Класс 4/10 (mc_id=106)

Класс 5/10 (mc_id=107)

Класс 6/10 (mc_id=108)

Класс 7/10 (mc_id=109)

Класс 8/10 (mc_id=110)

Класс 9/10 (mc_id=111)

📌 multilingual-e5-base — РЕЗУЛЬТАТЫ
F1 micro: 0.6826
F1 macro: 0.6783
logloss:  0.1112


(0.6825938566552902, 0.6782520999091757, np.float64(0.11122210991037848))

# **Нейронный пайплайн (encoder + multi‑label head + shouldSplit head) RuBERT‑base / ruBert‑base‑v2 / XLM‑R‑base**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import gc

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np

torch.cuda.empty_cache()
gc.collect()

df = pd.read_json("rnc_dataset_markup.json")

def parse_list(s):
    if isinstance(s, str) and s.startswith("["):
        return [int(x) for x in s.strip("[]").split(",") if x.strip()]
    return []

df["target"] = df["targetSplitMcIds"].apply(parse_list)
df["shouldSplit"] = df["target"].apply(lambda x: int(len(x) > 0))

all_mc_ids = sorted(set(mc for lst in df["target"] for mc in lst))
if 101 in all_mc_ids:
    all_mc_ids.remove(101)

mlb = MultiLabelBinarizer(classes=all_mc_ids)
Y_mc = mlb.fit_transform(df["target"])
Y_split = df["shouldSplit"].values
texts = df["description"].tolist()

X_train_texts, X_temp_texts, Y_mc_train, Y_mc_temp, Y_split_train, Y_split_temp = train_test_split(
    texts, Y_mc, Y_split,
    test_size=0.30,
    random_state=42,
    stratify=Y_split
)

X_val_texts, X_test_texts, Y_mc_val, Y_mc_test, Y_split_val, Y_split_test = train_test_split(
    X_temp_texts, Y_mc_temp, Y_split_temp,
    test_size=0.50,
    random_state=42,
    stratify=Y_split_temp
)


MAX_LEN = 256

class MCTextDataset(Dataset):
    def __init__(self, texts, y_mc, y_split, tokenizer):
        self.texts = texts
        self.y_mc = torch.tensor(y_mc, dtype=torch.float32)
        self.y_split = torch.tensor(y_split, dtype=torch.float32)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "y_mc": self.y_mc[idx],
            "y_split": self.y_split[idx]
        }


class MeanPooling(nn.Module):
    def forward(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        masked = last_hidden_state * mask
        summed = masked.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts


class TextEncoder(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        if hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()
        self.pool = MeanPooling()

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.pool(out.last_hidden_state, attention_mask)


class MultiLabelHead(nn.Module):
    def __init__(self, dim, n_classes, dropout=0.3):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, n_classes)
        )

    def forward(self, h):
        return self.fc(h)


class ShouldSplitHead(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, h):
        return self.fc(h)


class FullModel(nn.Module):
    def __init__(self, encoder_name, n_classes):
        super().__init__()
        self.encoder = TextEncoder(encoder_name)
        dim = self.encoder.encoder.config.hidden_size
        self.mc_head = MultiLabelHead(dim, n_classes)
        self.split_head = ShouldSplitHead(dim)

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids, attention_mask)
        return self.mc_head(h), self.split_head(h).squeeze(-1)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction="none")

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        pt = torch.exp(-bce_loss)
        focal = ((1 - pt) ** self.gamma) * bce_loss
        return focal.mean()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_classes = Y_mc.shape[1]

criterion_mc = FocalLoss(gamma=2.0)
criterion_split = nn.BCEWithLogitsLoss()

scaler = torch.amp.GradScaler("cuda")

accumulation_steps = 4

def run_epoch(model, loader, optimizer=None, scheduler=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0
    all_mc_true, all_mc_pred = [], []
    all_split_true, all_split_pred = [], []

    optimizer.zero_grad() if train_mode else None

    for step, batch in enumerate(loader):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y_mc = batch["y_mc"].to(device)
        y_split = batch["y_split"].to(device)

        with torch.set_grad_enabled(train_mode), torch.amp.autocast("cuda"):
            logits_mc, logit_split = model(ids, mask)
            loss = criterion_mc(logits_mc, y_mc) + criterion_split(logit_split, y_split)
            loss = loss / accumulation_steps

        if train_mode:
            scaler.scale(loss).backward()

            if (step + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                if scheduler:
                    scheduler.step()

        total_loss += loss.item() * ids.size(0)

        probs_mc = torch.sigmoid(logits_mc).detach().cpu().numpy()
        preds_mc = (probs_mc > 0.5).astype(int)

        probs_split = torch.sigmoid(logit_split).detach().cpu().numpy()
        preds_split = (probs_split > 0.5).astype(int)

        all_mc_true.append(y_mc.cpu().numpy())
        all_mc_pred.append(preds_mc)
        all_split_true.append(y_split.cpu().numpy())
        all_split_pred.append(preds_split)

        del ids, mask, y_mc, y_split, logits_mc, logit_split
        torch.cuda.empty_cache()

    all_mc_true = np.vstack(all_mc_true)
    all_mc_pred = np.vstack(all_mc_pred)
    all_split_true = np.concatenate(all_split_true)
    all_split_pred = np.concatenate(all_split_pred)

    return (
        total_loss / len(loader.dataset),
        f1_score(all_mc_true, all_mc_pred, average="micro", zero_division=0),
        f1_score(all_mc_true, all_mc_pred, average="macro", zero_division=0),
        f1_score(all_split_true, all_split_pred, average="binary", zero_division=0)
    )


def get_probs(model, loader):
    model.eval()
    all_probs = []
    all_true = []
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y_mc = batch["y_mc"].cpu().numpy()

            logits_mc, _ = model(ids, mask)
            probs = torch.sigmoid(logits_mc).detach().cpu().numpy()

            all_probs.append(probs)
            all_true.append(y_mc)

    return np.vstack(all_probs), np.vstack(all_true)


def find_best_threshold(model, val_loader):
    val_probs, val_true = get_probs(model, val_loader)

    thresholds = np.linspace(0.05, 0.50, 20)
    best_t, best_f1 = 0.5, 0.0

    for t in thresholds:
        preds = (val_probs > t).astype(int)
        f1 = f1_score(val_true, preds, average="micro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1


ENCODERS = [
    ("DeepPavlov/rubert-base-cased", "RuBERT-base"),
    ("ai-forever/ruBert-base", "ruBert-base-v2"),
    ("xlm-roberta-base", "XLM-R-base"),
]

EPOCHS = 7
BATCH_SIZE = 8


for encoder_name, label in ENCODERS:
    print("\n==============================")
    print(f"ENCODER: {label}")
    print("==============================")

    torch.cuda.empty_cache()
    gc.collect()

    tokenizer = AutoTokenizer.from_pretrained(encoder_name)

    train_ds = MCTextDataset(X_train_texts, Y_mc_train, Y_split_train, tokenizer)
    val_ds   = MCTextDataset(X_val_texts,   Y_mc_val,   Y_split_val,   tokenizer)
    test_ds  = MCTextDataset(X_test_texts,  Y_mc_test,  Y_split_test,  tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

    model = FullModel(encoder_name, n_classes).to(device)

    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": 2e-5, "weight_decay": 0.01},
        {"params": model.mc_head.parameters(), "lr": 5e-4, "weight_decay": 0.001},
        {"params": model.split_head.parameters(), "lr": 5e-4, "weight_decay": 0.001},
    ])

    total_steps = len(train_loader) * EPOCHS // accumulation_steps
    warmup_steps = int(total_steps * 0.1)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    for epoch in range(1, EPOCHS + 1):
        tr = run_epoch(model, train_loader, optimizer, scheduler)
        va = run_epoch(model, val_loader)

        print(f"\n[{label}] Epoch {epoch}/{EPOCHS}")
        print(f"Train loss={tr[0]:.4f}  F1_micro={tr[1]:.4f}  F1_macro={tr[2]:.4f}  F1_split={tr[3]:.4f}")
        print(f"Val   loss={va[0]:.4f}  F1_micro={va[1]:.4f}  F1_macro={va[2]:.4f}  F1_split={va[3]:.4f}")

    best_t, best_f1 = find_best_threshold(model, val_loader)
    print(f"\n[{label}] Лучший порог: {best_t:.3f}, F1_micro={best_f1:.4f}")

    test_probs, test_true = get_probs(model, test_loader)
    test_preds = (test_probs > best_t).astype(int)

    f1_micro = f1_score(test_true, test_preds, average="micro", zero_division=0)
    f1_macro = f1_score(test_true, test_preds, average="macro", zero_division=0)

    print(f"[{label}] TEST F1_micro={f1_micro:.4f}  F1_macro={f1_macro:.4f}")

    del model, optimizer, tokenizer, train_ds, val_ds, test_ds
    torch.cuda.empty_cache()
    gc.collect()



ENCODER: RuBERT-base


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[RuBERT-base] Epoch 1/7
Train loss=0.1566  F1_micro=0.0903  F1_macro=0.0828  F1_split=0.1339
Val   loss=0.1355  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0000

[RuBERT-base] Epoch 2/7
Train loss=0.1286  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0000
Val   loss=0.1257  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0000

[RuBERT-base] Epoch 3/7
Train loss=0.1131  F1_micro=0.0592  F1_macro=0.0483  F1_split=0.2902
Val   loss=0.1204  F1_micro=0.0106  F1_macro=0.0108  F1_split=0.4854

[RuBERT-base] Epoch 4/7
Train loss=0.0825  F1_micro=0.2350  F1_macro=0.1892  F1_split=0.6552
Val   loss=0.1330  F1_micro=0.1651  F1_macro=0.1390  F1_split=0.4182

[RuBERT-base] Epoch 5/7
Train loss=0.0529  F1_micro=0.4080  F1_macro=0.3315  F1_split=0.8220
Val   loss=0.1662  F1_micro=0.1933  F1_macro=0.1611  F1_split=0.4000

[RuBERT-base] Epoch 6/7
Train loss=0.0290  F1_micro=0.5752  F1_macro=0.4967  F1_split=0.9190
Val   loss=0.1950  F1_micro=0.2525  F1_macro=0.2211  F1_split=0.4483

[RuBERT-base] Epoch 7

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[ruBert-base-v2] Epoch 1/7
Train loss=0.1562  F1_micro=0.0750  F1_macro=0.0682  F1_split=0.0236
Val   loss=0.1386  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0000

[ruBert-base-v2] Epoch 2/7
Train loss=0.1299  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0060
Val   loss=0.1281  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0000

[ruBert-base-v2] Epoch 3/7
Train loss=0.1057  F1_micro=0.0868  F1_macro=0.0687  F1_split=0.3957
Val   loss=0.1204  F1_micro=0.1557  F1_macro=0.1206  F1_split=0.4673

[ruBert-base-v2] Epoch 4/7
Train loss=0.0649  F1_micro=0.3508  F1_macro=0.2901  F1_split=0.7358
Val   loss=0.1526  F1_micro=0.1762  F1_macro=0.1392  F1_split=0.4677

[ruBert-base-v2] Epoch 5/7
Train loss=0.0300  F1_micro=0.5764  F1_macro=0.5009  F1_split=0.9141
Val   loss=0.1948  F1_micro=0.3601  F1_macro=0.2963  F1_split=0.5161

[ruBert-base-v2] Epoch 6/7
Train loss=0.0146  F1_micro=0.6489  F1_macro=0.5582  F1_split=0.9802
Val   loss=0.2015  F1_micro=0.3495  F1_macro=0.2929  F1_split=0.5714

[ru

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[XLM-R-base] Epoch 1/7
Train loss=0.1552  F1_micro=0.0879  F1_macro=0.0602  F1_split=0.0000
Val   loss=0.1415  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0000

[XLM-R-base] Epoch 2/7
Train loss=0.1336  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0000
Val   loss=0.1338  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0000

[XLM-R-base] Epoch 3/7
Train loss=0.1258  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0119
Val   loss=0.1285  F1_micro=0.0000  F1_macro=0.0000  F1_split=0.0000

[XLM-R-base] Epoch 4/7
Train loss=0.1108  F1_micro=0.0225  F1_macro=0.0191  F1_split=0.2010
Val   loss=0.1240  F1_micro=0.0317  F1_macro=0.0261  F1_split=0.4444

[XLM-R-base] Epoch 5/7
Train loss=0.0917  F1_micro=0.2324  F1_macro=0.2010  F1_split=0.5534
Val   loss=0.1205  F1_micro=0.2554  F1_macro=0.2264  F1_split=0.4590

[XLM-R-base] Epoch 6/7
Train loss=0.0788  F1_micro=0.2630  F1_macro=0.2137  F1_split=0.6239
Val   loss=0.1353  F1_micro=0.2698  F1_macro=0.2299  F1_split=0.4672

[XLM-R-base] Epoch 7/7
Trai

# **Pipeline A (RuBERT‑base, per‑class thresholds)**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import gc

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np


torch.cuda.empty_cache()
gc.collect()


df = pd.read_json("rnc_dataset_markup.json")

def parse_list(s):
    if isinstance(s, str) and s.startswith("["):
        return [int(x) for x in s.strip("[]").split(",") if x.strip()]
    return []

df["target"] = df["targetSplitMcIds"].apply(parse_list)
df["shouldSplit"] = df["target"].apply(lambda x: int(len(x) > 0))

all_mc_ids = sorted(set(mc for lst in df["target"] for mc in lst))
if 101 in all_mc_ids:
    all_mc_ids.remove(101)

mlb = MultiLabelBinarizer(classes=all_mc_ids)
Y_mc = mlb.fit_transform(df["target"])
Y_split = df["shouldSplit"].values
texts = df["description"].tolist()

X_train_texts, X_temp_texts, Y_mc_train, Y_mc_temp, Y_split_train, Y_split_temp = train_test_split(
    texts, Y_mc, Y_split,
    test_size=0.30,
    random_state=42,
    stratify=Y_split
)

X_val_texts, X_test_texts, Y_mc_val, Y_mc_test, Y_split_val, Y_split_test = train_test_split(
    X_temp_texts, Y_mc_temp, Y_split_temp,
    test_size=0.50,
    random_state=42,
    stratify=Y_split_temp
)


MAX_LEN = 256

class MCTextDataset(Dataset):
    def __init__(self, texts, y_mc, y_split, tokenizer):
        self.texts = texts
        self.y_mc = torch.tensor(y_mc, dtype=torch.float32)
        self.y_split = torch.tensor(y_split, dtype=torch.float32)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "y_mc": self.y_mc[idx],
            "y_split": self.y_split[idx]
        }


class MeanPooling(nn.Module):
    def forward(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        masked = last_hidden_state * mask
        summed = masked.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts


class TextEncoder(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        if hasattr(self.encoder, "gradient_checkpointing_enable"):
            self.encoder.gradient_checkpointing_enable()
        self.pool = MeanPooling()

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.pool(out.last_hidden_state, attention_mask)


class MultiLabelHead(nn.Module):
    def __init__(self, dim, n_classes, dropout=0.3):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, n_classes)
        )

    def forward(self, h):
        return self.fc(h)


class ShouldSplitHead(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, h):
        return self.fc(h)


class FullModel(nn.Module):
    def __init__(self, encoder_name, n_classes):
        super().__init__()
        self.encoder = TextEncoder(encoder_name)
        dim = self.encoder.encoder.config.hidden_size
        self.mc_head = MultiLabelHead(dim, n_classes)
        self.split_head = ShouldSplitHead(dim)

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids, attention_mask)
        return self.mc_head(h), self.split_head(h).squeeze(-1)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction="none")

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        pt = torch.exp(-bce_loss)
        focal = ((1 - pt) ** self.gamma) * bce_loss
        return focal.mean()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_classes = Y_mc.shape[1]

criterion_mc = FocalLoss(gamma=2.0)
criterion_split = nn.BCEWithLogitsLoss()

scaler = torch.amp.GradScaler("cuda")
accumulation_steps = 4

def run_epoch(model, loader, optimizer=None, scheduler=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0
    all_mc_true, all_mc_pred = [], []
    all_split_true, all_split_pred = [], []

    if train_mode:
        optimizer.zero_grad()

    for step, batch in enumerate(loader):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y_mc = batch["y_mc"].to(device)
        y_split = batch["y_split"].to(device)

        with torch.set_grad_enabled(train_mode), torch.amp.autocast("cuda"):
            logits_mc, logit_split = model(ids, mask)
            loss = criterion_mc(logits_mc, y_mc) + criterion_split(logit_split, y_split)
            loss = loss / accumulation_steps

        if train_mode:
            scaler.scale(loss).backward()

            if (step + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                if scheduler:
                    scheduler.step()

        total_loss += loss.item() * ids.size(0)

        probs_mc = torch.sigmoid(logits_mc).detach().cpu().numpy()
        preds_mc = (probs_mc > 0.5).astype(int)

        probs_split = torch.sigmoid(logit_split).detach().cpu().numpy()
        preds_split = (probs_split > 0.5).astype(int)

        all_mc_true.append(y_mc.cpu().numpy())
        all_mc_pred.append(preds_mc)
        all_split_true.append(y_split.cpu().numpy())
        all_split_pred.append(preds_split)

        del ids, mask, y_mc, y_split, logits_mc, logit_split
        torch.cuda.empty_cache()

    all_mc_true = np.vstack(all_mc_true)
    all_mc_pred = np.vstack(all_mc_pred)
    all_split_true = np.concatenate(all_split_true)
    all_split_pred = np.concatenate(all_split_pred)

    return (
        total_loss / len(loader.dataset),
        f1_score(all_mc_true, all_mc_pred, average="micro", zero_division=0),
        f1_score(all_mc_true, all_mc_pred, average="macro", zero_division=0),
        f1_score(all_split_true, all_split_pred, average="binary", zero_division=0)
    )


def get_probs(model, loader):
    model.eval()
    all_probs = []
    all_true = []
    with torch.no_grad(), torch.amp.autocast("cuda"):
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y_mc = batch["y_mc"].cpu().numpy()

            logits_mc, _ = model(ids, mask)
            probs = torch.sigmoid(logits_mc).detach().cpu().numpy()

            all_probs.append(probs)
            all_true.append(y_mc)

    return np.vstack(all_probs), np.vstack(all_true)

def find_per_class_thresholds(val_probs, val_true):
    n_classes = val_true.shape[1]
    thresholds = np.zeros(n_classes)
    thresholds[:] = 0.5
    thresholds_grid = np.linspace(0.05, 0.50, 20)

    for i in range(n_classes):
        best_t = 0.5
        best_f1 = 0.0
        y_true_i = val_true[:, i]
        if y_true_i.sum() == 0:
            continue
        for t in thresholds_grid:
            y_pred_i = (val_probs[:, i] > t).astype(int)
            f1 = f1_score(y_true_i, y_pred_i, average="binary", zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        thresholds[i] = best_t
    return thresholds

ENCODERS = [
    ("DeepPavlov/rubert-base-cased", "RuBERT-base"),
]

EPOCHS = 7
BATCH_SIZE = 8
PATIENCE = 3


for encoder_name, label in ENCODERS:
    print("\n==============================")
    print(f"PIPELINE A — PER-CLASS THRESHOLDS")
    print(f"ENCODER: {label}")
    print("==============================")

    torch.cuda.empty_cache()
    gc.collect()

    tokenizer = AutoTokenizer.from_pretrained(encoder_name)

    train_ds = MCTextDataset(X_train_texts, Y_mc_train, Y_split_train, tokenizer)
    val_ds   = MCTextDataset(X_val_texts,   Y_mc_val,   Y_split_val,   tokenizer)
    test_ds  = MCTextDataset(X_test_texts,  Y_mc_test,  Y_split_test,  tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

    model = FullModel(encoder_name, n_classes).to(device)

    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": 2e-5, "weight_decay": 0.01},
        {"params": model.mc_head.parameters(), "lr": 5e-4, "weight_decay": 0.001},
        {"params": model.split_head.parameters(), "lr": 5e-4, "weight_decay": 0.001},
    ])

    total_steps = len(train_loader) * EPOCHS // accumulation_steps
    warmup_steps = int(total_steps * 0.1)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_val_f1_micro = 0.0
    best_state_dict = None
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        tr = run_epoch(model, train_loader, optimizer, scheduler)
        va = run_epoch(model, val_loader)

        print(f"\nEpoch {epoch}/{EPOCHS}")
        print(f"Train loss={tr[0]:.4f}  F1_micro={tr[1]:.4f}  F1_macro={tr[2]:.4f}")
        print(f"Val   loss={va[0]:.4f}  F1_micro={va[1]:.4f}  F1_macro={va[2]:.4f}")

        val_f1_micro = va[1]
        if val_f1_micro > best_val_f1_micro:
            best_val_f1_micro = val_f1_micro
            best_state_dict = {k: v.cpu() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping")
                break

    if best_state_dict is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state_dict.items()})

    val_probs, val_true = get_probs(model, val_loader)
    per_class_t = find_per_class_thresholds(val_probs, val_true)

    test_probs, test_true = get_probs(model, test_loader)
    test_preds = (test_probs > per_class_t).astype(int)

    f1_micro = f1_score(test_true, test_preds, average="micro", zero_division=0)
    f1_macro = f1_score(test_true, test_preds, average="macro", zero_division=0)

    print(f"\nPIPELINE A — TEST RESULTS")
    print(f"F1_micro={f1_micro:.4f}  F1_macro={f1_macro:.4f}")

    del model, optimizer, tokenizer, train_ds, val_ds, test_ds
    torch.cuda.empty_cache()
    gc.collect()



PIPELINE A — PER-CLASS THRESHOLDS
ENCODER: RuBERT-base


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/7
Train loss=0.1599  F1_micro=0.0560  F1_macro=0.0496
Val   loss=0.1342  F1_micro=0.0000  F1_macro=0.0000

Epoch 2/7
Train loss=0.1298  F1_micro=0.0080  F1_macro=0.0063
Val   loss=0.1311  F1_micro=0.0000  F1_macro=0.0000

Epoch 3/7
Train loss=0.1110  F1_micro=0.0891  F1_macro=0.0720
Val   loss=0.1228  F1_micro=0.0159  F1_macro=0.0150

Epoch 4/7
Train loss=0.0844  F1_micro=0.3079  F1_macro=0.2619
Val   loss=0.1368  F1_micro=0.0798  F1_macro=0.0725

Epoch 5/7
Train loss=0.0582  F1_micro=0.4105  F1_macro=0.3436
Val   loss=0.1355  F1_micro=0.2840  F1_macro=0.2392

Epoch 6/7
Train loss=0.0319  F1_micro=0.5607  F1_macro=0.4836
Val   loss=0.1702  F1_micro=0.2966  F1_macro=0.2606

Epoch 7/7
Train loss=0.0227  F1_micro=0.6235  F1_macro=0.5518
Val   loss=0.1729  F1_micro=0.3027  F1_macro=0.2701

PIPELINE A — TEST RESULTS
F1_micro=0.4711  F1_macro=0.4559
